In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
from pathlib import Path
sys.path.append("..") 
# sys.path.append("C:/Users/ngoak/OneDrive/Projects/agentic")

from typing import List

import numpy as np
import cv2
import matplotlib.pyplot as plt
import PIL

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn.functional as F

from transformers import CLIPSegConfig, CLIPSegProcessor
from transformers import CLIPSegForImageSegmentation

from configuration_refiner import RefinerConfig
from modeling_refiner import Refiner

from accelerate import Accelerator
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers.models.grounding_dino.modeling_grounding_dino import GroundingDinoObjectDetectionOutput as DetectionOutput

from transformers import Sam2Model, Sam2VideoModel, Sam2VideoProcessor
from transformers.models.sam2.modeling_sam2 import Sam2ImageSegmentationOutput as SegmentationOutput

from kitti_tracking import KittiDataset
from DSTT.DSTT_OM import InpaintGenerator
from inpainting import InpaintingTool

In [ ]:
# CLIPSeg
model_name = "CIDAS/clipseg-rd64-refined"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clipseg_config = CLIPSegConfig.from_pretrained(model_name)
clipseg_processor = CLIPSegProcessor.from_pretrained(model_name, use_fast=True)
segm_model = CLIPSegForImageSegmentation.from_pretrained(model_name).to(device)

In [ ]:
# refiner
refiner_config = RefinerConfig(
    action_channels=2,
    **clipseg_config.to_dict()
)
model = Refiner._from_config(refiner_config)

In [ ]:
# DSTT
painting_pre_process = transforms.Compose([
	transforms.Resize((240, 432)),
    transforms.ToTensor(),      # scales to [0, 1]
    transforms.Lambda(lambda x: 2 * x - 1)  # scales to [-1, 1]
])

ref_step, num_refs, num_neighbors = 10, 3, 3
inpainting_memory = torch.Tensor().to(device)

In [ ]:
def _collate_fn(data):
    image_list, label_list = zip(*data)
    return image_list, label_list

root_dir = "E:/KittiTracking"
train_ds = KittiDataset(root_dir, "train", 8, 3)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=_collate_fn)

In [ ]:
dstt_transforms=transforms.Compose([
	transforms.Resize((240, 432)),
	transforms.ToTensor(),      # scales to [0, 1]
	transforms.Lambda(lambda x: 2 * x - 1)  # scales to [-1, 1]
])

In [ ]:
prompt = ["car"]
dstt_w, dstt_h = 432, 240
n_steps, n_pred_steps = 8, 3

for batch in train_loader:
	imgs, labels = batch
	b = len(imgs)
	w, h = imgs[0][0].size

	# 1. generate masked frames
	# NOTE: randomly omit some objects
	
	# Preprocess
	imgs_flatten = [_img for img in imgs for _img in img[:n_steps]]

	# Inference
	with torch.no_grad():
		inputs = clipseg_processor(text=b*n_steps*prompt, images=imgs_flatten, return_tensors="pt").to(device)
		outputs = segm_model(**inputs)
		
		# extract semantic mask
		masks = outputs.logits.sigmoid().float().cpu().numpy()
		sem_masks = [
			PIL.Image.fromarray(255*np.uint8(mask>0.5), "L").resize(img.size)
			for img, mask in zip(imgs_flatten, masks)
		]

		# 
		masked_imgs_flatten = [
			PIL.Image.composite(img, PIL.Image.new("RGB", img.size), mask)
			for img, mask in zip(imgs_flatten, sem_masks)
		]
		masked_imgs = [[
				masked_imgs_flatten[i*n_steps + j]
				for j in range(n_steps)
			] for i in range(b)
		]

		# 2. reconstruct full frames
		# resize
		dstt_imgs = [
			_img.resize((round(img.size[0]/dstt_w+0.1)*dstt_w, round(img.size[1]/dstt_h+0.1)*dstt_h))
			for img in imgs
			for _img in img
		]
		# for step in range(n_steps+1):
		# 	# recon_frames = dstt([])
		# 	pass

	inp_imgs = [img[n_steps+1] for img in imgs]
	inputs = clipseg_processor(text=b*prompt, images=inp_imgs, return_tensors="pt").to(device)
	outputs = segm_model(**inputs)

	break

# imgs.shape, labels.shape

In [ ]:
len(imgs), len(imgs[0])

In [ ]:
round(640/dstt_w+0.1)*dstt_w, round(480/dstt_h+0.1)*dstt_h

In [ ]:
device = Accelerator().device

obj_det_model_id = "IDEA-Research/grounding-dino-tiny"
obj_det_processor = AutoProcessor.from_pretrained(obj_det_model_id)
obj_det_model = AutoModelForZeroShotObjectDetection.from_pretrained(obj_det_model_id).eval().to(device, dtype=torch.bfloat16)
for p in obj_det_model.parameters(): p.requires_grad_(False)

sam_model_id = "facebook/sam2.1-hiera-tiny"
sam_processor = Sam2VideoProcessor.from_pretrained(sam_model_id)
sam_model = Sam2Model.from_pretrained(sam_model_id).to(device, dtype=torch.bfloat16)
for p in sam_model.parameters(): p.requires_grad_(False)

In [ ]:
prompt = ["car"]
dstt_w, dstt_h = 432, 240
n_steps, n_pred_steps = 16, 3

dumb_box = [[0., 0., 0., 0.]]
for batch in train_loader:
	imgs, labels = batch
	b = len(imgs)
	w, h = imgs[0][0].size

	# 1. generate masked frames
	# NOTE: randomly omit some objects
	
	# Preprocess
	imgs_flatten = [_img for img in imgs for _img in img[:n_steps]]

	# Inference
	with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
		# GroundDINO
		inputs = obj_det_processor(images=imgs_flatten, text=[prompt for _ in imgs_flatten], return_tensors="pt").to(device)
		outputs: DetectionOutput = obj_det_model(**inputs)
		results: List = obj_det_processor.post_process_grounded_object_detection(
			outputs,
			inputs.input_ids,
			threshold=0.4,
			text_threshold=0.3,
			target_sizes=[img.size[::-1] for img in imgs_flatten]
		)
		boxes = [result["boxes"].cpu().tolist() for result in results]

		# SAM
		max_len = max([len(bbox) for bbox in boxes])
		_boxes = [box + dumb_box * (max_len - len(box)) for box in boxes]
		inputs = sam_processor(images=imgs_flatten, input_boxes=_boxes, return_tensors="pt").to(device)
		outputs = sam_model(**inputs, multimask_output=False)

		masks = outputs.pred_masks.sigmoid().float().cpu().numpy()

		sem_masks = []
		for mask in masks:
			combined_mask = np.zeros((1, *imgs_flatten[0].size[::-1]), dtype=np.uint8)
			for m in mask:
				_mask = PIL.Image.fromarray(np.uint8(m[0] > 0.5), "L").resize(imgs_flatten[0].size),
				combined_mask = np.maximum(np.array(_mask), combined_mask)
			sem_mask = PIL.Image.fromarray(255*combined_mask[0], "L")
			sem_masks.append(sem_mask)

	break


In [ ]:
imgs_flatten[16]

In [ ]:
sem_masks = []
for mask in masks:
	combined_mask = np.zeros((1, *imgs_flatten[0].size[::-1]), dtype=np.uint8)
	for m in mask:
		_mask = PIL.Image.fromarray(np.uint8(m[0] > 0.1), "L").resize(imgs_flatten[0].size),
		combined_mask = np.maximum(np.array(_mask), combined_mask)
	sem_mask = PIL.Image.fromarray(255*combined_mask[0], "L")
	sem_masks.append(sem_mask)

sem_masks[16]

In [ ]:
draw = PIL.ImageDraw.Draw(imgs_flatten[0])
for box, score, label in zip(results[0]["boxes"], results[0]["scores"], results[0]["text_labels"]):
    x0, y0, x1, y1 = box.tolist()
    draw.rectangle([x0, y0, x1, y1], outline="red", width=3)
    draw.text((x0, y0), f"{label} {score:.2f}", fill="red")
    
# Show/save
imgs_flatten[0].show()
# imgs_flatten[0].save("./output_with_boxes.jpg")

In [ ]:

masks = outputs.pred_masks.sigmoid().float().cpu().numpy()

image = np.asarray(imgs_flatten)

H, W = image.shape[:2]
overlay_img = image.copy().astype(np.float32)

combined_mask = np.zeros((1, *imgs_flatten[0].size[::-1]), dtype=np.uint8)
for i, mask in enumerate(masks):
	_mask = PIL.Image.fromarray(np.uint8(mask[0]), "L").resize(imgs_flatten[0].size),
	combined_mask = np.maximum(np.array(_mask), combined_mask)

sem_masks = PIL.Image.fromarray(255*combined_mask[0], "L")

plt.imshow(image)
plt.axis("off")

In [ ]:
sem_mask

In [ ]:
plt.imshow(
    combined_mask[0],
    cmap="grey"
)
plt.axis("off")
plt.show()